In [1]:
%pip install "numpy<2.0.0" --force-reinstall

Note: you may need to restart the kernel to use updated packages.Collecting numpy<2.0.0
  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4



  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [2]:
import pandas as pd
import glob
import os

print("=== INICIO DE COMPILACIÓN Y DESGLOSE POR TILTMETER ===")

# 1. Localizar los archivos dentro de Rail y Sleeper (formatos .csv y .txt)
archivos_rail = glob.glob("Rail/*.csv") + glob.glob("Rail/*.txt")
archivos_sleeper = glob.glob("Sleeper/*.csv") + glob.glob("Sleeper/*.txt")

# Búsqueda secundaria de respaldo por si existen subcarpetas o minúsculas
if not archivos_rail:
    archivos_rail = [f for f in glob.glob("Rail/**/*", recursive=True) if f.endswith(('.csv', '.txt'))]
if not archivos_sleeper:
    archivos_sleeper = [f for f in glob.glob("Sleeper/**/*", recursive=True) if f.endswith(('.csv', '.txt'))]

print(f"Archivos encontrados en Rail: {len(archivos_rail)}")
print(f"Archivos encontrados en Sleeper: {len(archivos_sleeper)}")

def procesar_y_desglosar_archivo(ruta, posicion_via):
    """
    Lee un CSV/TXT crudo de Worldsensing, detecta las columnas repetidas
    por cada tiltmeter y reconstruye la serie temporal desglosada.
    """
    try:
        # Detectar separador (coma o punto y coma) leyendo las primeras líneas
        with open(ruta, 'r', encoding='utf-8', errors='ignore') as f:
            first_line = f.readline()
        sep = ';' if ';' in first_line else ','
        
        df_raw = pd.read_csv(ruta, sep=sep, dtype=str, on_bad_lines='skip')
        df_raw = df_raw.dropna(how='all', axis=1)
        
        if df_raw.empty:
            return None

        # Identificar la columna de marca de tiempo (timestamp / date / time)
        col_time = [c for c in df_raw.columns if any(k in str(c).lower() for k in ['date', 'time', 'fecha', 'timestamp'])]
        
        if col_time:
            timestamp_series = pd.to_datetime(df_raw[col_time[0]], errors='coerce')
        else:
            # Si no hay timestamp explícito, se usa una secuencia temporal simulada
            timestamp_series = pd.date_range(start="2026-01-01", periods=len(df_raw), freq="15min")

        # Detectar si hay múltiples tiltmeters distribuidos en las columnas
        cols_x = [c for c in df_raw.columns if any(k in str(c).lower() for k in ['axis_x', 'x_reading', 'x_deg', 'x('])]
        cols_y = [c for c in df_raw.columns if any(k in str(c).lower() for k in ['axis_y', 'y_reading', 'y_deg', 'y('])]
        cols_z = [c for c in df_raw.columns if any(k in str(c).lower() for k in ['axis_z', 'z_reading', 'z_deg', 'z('])]
        cols_t = [c for c in df_raw.columns if any(k in str(c).lower() for k in ['temp', 'temperature'])]

        num_tiltmeters = max(len(cols_x), len(cols_y), len(cols_z), 1)
        bloques_tiltmeters = []

        # Extraer cada bloque de sensor de forma independiente
        for i in range(num_tiltmeters):
            df_sensor = pd.DataFrame()
            df_sensor['timestamp'] = timestamp_series
            
            # Asignar lecturas X, Y, Z y Temperatura asociadas al sensor i
            df_sensor['X_reading_deg'] = pd.to_numeric(df_raw[cols_x[i]], errors='coerce') if i < len(cols_x) else 0.0
            df_sensor['Y_reading_deg'] = pd.to_numeric(df_raw[cols_y[i]], errors='coerce') if i < len(cols_y) else 0.0
            df_sensor['Z_reading_deg'] = pd.to_numeric(df_raw[cols_z[i]], errors='coerce') if i < len(cols_z) else 0.0
            df_sensor['temperature_C'] = pd.to_numeric(df_raw[cols_t[i]], errors='coerce') if i < len(cols_t) else 20.0
            
            # Etiquetar identificador del sensor y posición en la vía
            nombre_sensor = f"{posicion_via}_Tiltmeter_{i+1}"
            df_sensor['sensor_id'] = nombre_sensor
            df_sensor['row_status'] = 'OK'
            df_sensor['incident_id'] = 0
            
            bloques_tiltmeters.append(df_sensor)

        # Concatenar todos los tiltmeters extraídos del archivo actual
        return pd.concat(bloques_tiltmeters, ignore_index=True)

    except Exception as e:
        print(f"Error procesando el archivo {ruta}: {e}")
        return None

# 2. PROCESAR TODOS LOS ARCHIVOS Y APILAR
lista_dfs = []

for f in archivos_rail:
    df_res = procesar_y_desglosar_archivo(f, 'Rail')
    if df_res is not None:
        lista_dfs.append(df_res)

for f in archivos_sleeper:
    df_res = procesar_y_desglosar_archivo(f, 'Sleeper')
    if df_res is not None:
        lista_dfs.append(df_res)

# 3. CONCATENAR, ORDENAR Y CONSOLIDAR EN EL CSV FINAL
if lista_dfs:
    df_final = pd.concat(lista_dfs, ignore_index=True)
    
    # Ordenar cronológicamente las series temporales por timestamp
    df_final = df_final.sort_values('timestamp').reset_index(drop=True)
    
    # Imputación de nulos por propagación temporal
    df_final = df_final.ffill().bfill().fillna(0)
    
    # Formatear la columna de timestamp a cadena ISO estándar
    df_final['timestamp'] = df_final['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

    # Guardar en data/
    os.makedirs("data", exist_ok=True)
    ruta_salida = "data/tiltmonitor_incidence_series_24h.csv"
    df_final.to_csv(ruta_salida, index=False)
    
    print("\n==================================================")
    print(" ¡PROCESAMIENTO COMPLETO Y EXITOSO! ")
    print("==================================================")
    print(f" Archivo final guardado en: {os.path.abspath(ruta_salida)}")
    print(f" Total registros compilados: {df_final.shape[0]}")
    print(f" Total de columnas: {df_final.shape[1]}")
    print(f" Valores nulos restantes (NaN): {df_final.isna().sum().sum()}")
    print(f" Sensores desglosados únicos: {df_final['sensor_id'].nunique()}")
    print("==================================================")
else:
    print("No se pudieron procesar datos. Revisa que las carpetas Rail/ y Sleeper/ contengan archivos válidos.")

=== INICIO DE COMPILACIÓN Y DESGLOSE POR TILTMETER ===
Archivos encontrados en Rail: 26
Archivos encontrados en Sleeper: 26

 ¡PROCESAMIENTO COMPLETO Y EXITOSO! 
 Archivo final guardado en: c:\Users\maryp\Desktop\Carrera\practicas\prueba_world\data\tiltmonitor_incidence_series_24h.csv
 Total registros compilados: 1724934
 Total de columnas: 8
 Valores nulos restantes (NaN): 0
 Sensores desglosados únicos: 84
